### Random Image inference
(from an already trained model)

In [ ]:
from diffusers import DiffusionPipeline
import torch

# Load the pretrained pipeline
pipeline = DiffusionPipeline.from_pretrained("WaveDave/trial_6").to("cuda")

# Loop to generate 30 images
for i in range(30):
    # Generate the image
    image = pipeline().images[0]
    
    # Save the image with a unique filename
    image.save(f'C:/code_DG/diffusers-fork/venv_dg_uncondiffusion_fork/trial_6/inference_images/infer_test{i+1}.png')

-------------------------------------------------------------------------------------------------------------------------------------------

### Image Reconstruction via Denoising from Noise

1. Load your trained model

In [1]:
from diffusers import DDPMPipeline
from diffusers import DiffusionPipeline
import torch
from PIL import Image
import torchvision.transforms as T

pipeline = DiffusionPipeline.from_pretrained("WaveDave/trial_6").to("cuda")


c:\code_DG\diffusers-fork\venv_dg_uncondiffusion_fork\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 2/2 [00:00<00:00, 25.97it/s]


2. prepare input image 

- resize to 256x256, convert to grayscale, normalise to [-1,1]

In [10]:
image_path = "C:/code_DG/diffusers-fork/saved_recons/recon_7788.png"

transform = T.Compose([
    T.Resize((256,256)),
    T.Grayscale(),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]) # Match training normalisation 
])

original_image = Image.open(image_path).convert("L")
image_tensor = transform(original_image).unsqueeze(0).to("cuda") # Shape: [1, 1, 256, 256]

3. add noise at a given timestep

- choose how far along the noise process you want to inject the image:

In [11]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler

scheduler = pipeline.scheduler
timestep = 500 # halfway through the diffusion process 

# sample random noise 
noise = torch.randn_like(image_tensor)

# add noise to the image 
noisy_image = scheduler.add_noise(image_tensor, noise, torch.tensor([timestep]).to("cuda"))

4. denoise from the noisy image

manually denoise from timestep 500 down to 0:

In [12]:
from tqdm import tqdm

latents = noisy_image

for t in tqdm(reversed(range(timestep))):
    timesteps = torch.tensor([t], device="cuda")
    with torch.no_grad():
        noise_pred = pipeline.unet(latents, timesteps).sample
    latents = scheduler.step(noise_pred, t, latents).prev_sample

500it [00:11, 42.09it/s]


5. Convert output back to image

In [13]:
output = latents.detach().cpu().squeeze(0)
output = (output * 0.5 + 0.5).clamp(0, 1) # De-normalise 
output_image = T.ToPILImage()(output)
output_image.save("reconstructed_image_DIFFUSED_7788.png")